# 04 — Détection d'anomalies (non-supervisée)

Le split temporel (notebook 03) a montré que la détection supervisée
s'effondre sur les attaques inédites (recall 0.08). On teste ici une
approche non-supervisée : l'Isolation Forest apprend uniquement la
"normalité" du trafic et signale toute déviation — sans jamais voir
d'attaque. Objectif : mieux généraliser aux attaques inconnues (zero-day).

In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet("../data/processed/cicids_clean.parquet")
df.shape

(2827876, 81)

## 1. Construction du jeu de normalité

On isole le trafic BENIGN uniquement : le modèle doit apprendre à quoi
ressemble un flux normal, sans contamination par des attaques.

In [3]:
df_safe = df[ df["is_attack"] == 0 ]

In [4]:
df_safe_clean = df_safe.drop(columns=["Label", "is_attack", "source_day"])

In [5]:
df_safe_clean.shape

(2271320, 78)

## 2. Libération mémoire

df et df_safe ne sont plus utiles après extraction des features. On les
supprime pour libérer la RAM avant l'entraînement (contrainte machine 8 Go).

In [8]:
del df, df_safe
import gc
gc.collect()

0

## 3. Isolation Forest

Principe : construire des arbres qui découpent les données au hasard. Un
point anormal, situé à l'écart, s'isole en peu de découpes ; un point
normal, noyé dans la masse, en demande beaucoup. Anomalie = point facile
à isoler.

Avantage mémoire : chaque arbre ne s'entraîne que sur 256 points
(max_samples='auto'), ce qui permet de traiter les 2,27M lignes complètes
là où Random Forest saturait.

In [6]:
from sklearn.ensemble import IsolationForest

In [7]:
IF_model = IsolationForest( 
    contamination = 'auto' ,
    n_estimators = 100 ,
    random_state = 42 ,
    n_jobs = -1
)

## 4. Entraînement non-supervisé

`.fit()` ne reçoit QUE les features, aucun label — c'est tout l'esprit du
non-supervisé. Le modèle n'apprend pas "ceci est une attaque", il apprend
la structure du trafic normal.

In [9]:
IF_model.fit(df_safe_clean)

,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary <n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",100
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",'auto'
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary <warm_start>`... versionadded:: 0.21",False
Name,Type,Value
estimator_ estimator_: :class:`~sklearn.tree.ExtraTreeRegressor` instanceThe child estimator template used to create the collection offitted sub-estimators... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,ExtraTreeRegressor,ExtraTreeRegr...ndom_state=42)


## 5. Jeu de test (normal + attaques)

df a été supprimé, on le recharge. Le test est un échantillon du dataset
complet : ~80% normal + ~20% attaques. is_attack sert UNIQUEMENT à
l'évaluation finale — le modèle ne l'a jamais vu.

In [10]:
df = pd.read_parquet("../data/processed/cicids_clean.parquet")

In [11]:
df_test = df.sample(n=300000, random_state=42)
X_test = df_test.drop(columns=["Label", "is_attack", "source_day"])
y_test = df_test["is_attack"]

## 6. Prédiction et conversion

L'Isolation Forest renvoie 1 (normal) et -1 (anomalie) — conventions
différentes de notre 0/1. On convertit : anomalie (-1) → 1 (attaque),
normal (1) → 0 (BENIGN).

In [12]:
y_pred_if = IF_model.predict(X_test)

In [19]:
# -1 (anomalie) -> 1 (attaque) ; 1 (normal) -> 0 (BENIGN)
y_pred_converted = (y_pred_if == -1).astype(int)

In [13]:
from sklearn.metrics import classification_report

In [17]:
print(classification_report(y_test,y_pred_converted))

              precision    recall  f1-score   support

           0       0.87      0.92      0.90    240779
           1       0.59      0.45      0.51     59221

    accuracy                           0.83    300000
   macro avg       0.73      0.69      0.70    300000
weighted avg       0.82      0.83      0.82    300000



### Isolation Forest — détection non-supervisée

Le modèle est entraîné uniquement sur du trafic BENIGN (aucune attaque vue),
puis évalué sur un mélange normal + attaques.

| Approche | Recall (ATTACK) | Precision (ATTACK) |
|----------|-----------------|--------------------|
| Supervisé, split temporel | 0.08 | 0.99 |
| Isolation Forest (non-supervisé) | 0.45 | 0.59 |

Sans avoir jamais observé d'attaque, l'Isolation Forest en détecte 45% —
5× plus que le modèle supervisé confronté à des attaques inédites (0.08).
Il paie ce gain par une precision plus faible (plus de faux positifs).

Conclusion : le non-supervisé généralise mieux aux attaques inconnues
(zero-day), le supervisé est plus précis sur les attaques connues. Un NIDS
robuste combine les deux approches.

Note : la sortie de l'Isolation Forest (1 = normal, -1 = anomalie) a été
convertie vers la convention 0/1 pour l'évaluation.